### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "landuse.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/landuse.geojson",
    "protected_zones.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/protected_zones.geojson",
    "study_area.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/study_area.geojson"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp10" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP10 — Clip / Intersection / Overlay

Περιορισμός σε περιοχή μελέτης και intersection δύο polygon layers. Πριν γράψεις κώδικα αποφάσισε αν θέλεις μεταφορά attributes (spatial join) ή νέα geometry (clip/overlay). Το clip περιορίζει, το intersection δημιουργεί κοινή geometry.

In [ ]:
import geopandas as gpd
from pathlib import Path
root = DATA_ROOT / "gp10"
study = gpd.read_file(root / 'study_area.geojson')
landuse = gpd.read_file(root / 'landuse.geojson')
protected = gpd.read_file(root / 'protected_zones.geojson')


In [ ]:
target_crs = 'EPSG:2100'
study = study.to_crs(target_crs)
landuse = landuse.to_crs(target_crs)
protected = protected.to_crs(target_crs)


In [ ]:
clipped = gpd.clip(landuse, study)
overlay = gpd.overlay(clipped, protected, how='intersection')
clipped
overlay


Έλεγξε CRS, rows, geometry types, validity, empty geometries, bounds και area πριν από export. Θυμήσου: το clip μπορεί να κόψει geometry, το overlay μπορεί να αυξήσει rows και να μεταφέρει attributes και από τα δύο inputs.

In [ ]:
clipped['area_m2'] = clipped.geometry.area
overlay['area_m2'] = overlay.geometry.area
for frame in (study, clipped, overlay):
    assert frame.crs.to_epsg() == 2100
    assert frame.geometry.notna().all()
    assert frame.geometry.is_empty.sum() == 0
    assert frame.geometry.is_valid.all()
assert (clipped['area_m2'] > 0).all()
assert (overlay['area_m2'] > 0).all()
print('areas:', clipped['area_m2'].sum(), overlay['area_m2'].sum())


In [ ]:
clipped.to_file('gp10_results.gpkg', layer='clipped_landuse', driver='GPKG')
overlay.to_file('gp10_results.gpkg', layer='landuse_protected', driver='GPKG')
print('export complete')
